In [3]:
import polars as pl
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.loader import NeighborLoader
import numpy as np
import time
import gc
from tqdm import tqdm

# 1. 경로 설정 (파일에서 참조)
raw_path = "/home/tracerofjageum/HI-Medium_Master.parquet"
v2_path = "/home/tracerofjageum/aml_features_v2_integrated_final.parquet"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---------------------------------------------------------
# 2. 데이터 로드 및 graph_data 객체 생성 (기존 로직 복구)
# ---------------------------------------------------------
print("📂 기초 데이터 로딩 및 그래프 구축 시작...")
df_raw = pl.read_parquet(raw_path)
df_v2 = pl.read_parquet(v2_path).with_columns(pl.col("account_id").cast(pl.Utf8))

# 시간 기준 설정 (6:2:2 분할)
df_v2 = df_v2.sort("time_group")
train_idx = int(len(df_v2) * 0.6)
train_cutoff = df_v2["time_group"][train_idx]

# 노드 매핑
all_accounts = pl.concat([
    df_raw.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id")),
    df_raw.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
]).unique().with_row_index("node_id")

# 엣지 생성 (Train 기간 데이터만 사용)
df_raw_train = df_raw.filter(pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False) < train_cutoff)
df_edges = df_raw_train.select(["from_acc", "to_acc"]).with_columns([pl.col("from_acc").cast(pl.Utf8), pl.col("to_acc").cast(pl.Utf8)])
df_edges = df_edges.join(all_accounts.rename({"account_id": "from_acc", "node_id": "src_id"}), on="from_acc", how="left")
df_edges = df_edges.join(all_accounts.rename({"account_id": "to_acc", "node_id": "dst_id"}), on="to_acc", how="left")
edge_index = torch.tensor(df_edges.select(["src_id", "dst_id"]).to_numpy().T, dtype=torch.long)

# 노드 피처 및 라벨 생성
exclude_cols = ["account_id", "time_group", "is_laundering", "mode_format", "currency_mode"]
gnn_feature_cols = [c for c in df_v2.columns if c not in exclude_cols]
df_v2_train = df_v2.filter(pl.col("time_group") < train_cutoff)
df_v2_node = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feature_cols])
df_node_features = all_accounts.join(df_v2_node, on="account_id", how="left").fill_null(0.0)

X = torch.tensor(df_node_features.select(gnn_feature_cols).to_numpy(), dtype=torch.float32)
target_labels = df_v2_train.filter(pl.col("is_laundering") == 1).select("account_id").unique().with_columns(pl.lit(1).alias("label"))
Y = torch.tensor(all_accounts.join(target_labels, on="account_id", how="left").fill_null(0)["label"].to_numpy(), dtype=torch.long)

# 마스크 설정
active_accounts = df_v2_train["account_id"].unique()
train_mask = torch.tensor(all_accounts.join(pl.DataFrame({"account_id": active_accounts, "is_active": True}), on="account_id", how="left").fill_null(False)["is_active"].to_numpy(), dtype=torch.bool)

# 최종 graph_data 생성
graph_data = Data(x=X, edge_index=edge_index, y=Y)
graph_data.train_mask = train_mask

print(f"✅ graph_data 준비 완료: {graph_data}")

# ---------------------------------------------------------
# 3. 실험용 모델 클래스 정의
# ---------------------------------------------------------
class GraphSAGE_Universal_Exp(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, aggr_type='mean'):
        super(GraphSAGE_Universal_Exp, self).__init__()
        # aggr_type: 'mean', 'max', 'lstm'
        self.conv1 = SAGEConv(in_channels, hidden_channels, aggr=aggr_type)
        self.conv2 = SAGEConv(hidden_channels, 1, aggr=aggr_type)

    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=0.3, training=self.training)
        return self.conv2(h, edge_index)

# ---------------------------------------------------------
# 4. 집계 방식별 실험 루프
# ---------------------------------------------------------
aggr_list = ['mean', 'max', 'lstm']
train_loader = NeighborLoader(graph_data, num_neighbors=[15, 10], batch_size=2048, input_nodes=graph_data.train_mask, shuffle=True)

for aggr in aggr_list:
    print(f"\n🚀 [실험 중] 현재 집계 방식: {aggr.upper()}")
    
    
    torch.manual_seed(42) # 재현성을 위한 시드 고정
    start_time = time.time()
    
    model = GraphSAGE_Universal_Exp(graph_data.num_features, 64, aggr_type=aggr).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([48.0], device=device))
    
    model.train()
    for epoch in range(1, 11): # 빠른 비교를 위해 10 Epoch만 실행
        total_loss = 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index).squeeze(-1)
            loss = criterion(out[:batch.batch_size], batch.y[:batch.batch_size].float())
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"  Epoch {epoch:02d} | Loss: {total_loss/len(train_loader):.4f}")
            
    elapsed = time.time() - start_time
    print(f"✅ {aggr.upper()} 학습 완료! (소요 시간: {elapsed:.2f}초)")
    
    # 메모리 정리
    del model, optimizer; torch.cuda.empty_cache(); gc.collect()

📂 기초 데이터 로딩 및 그래프 구축 시작...
✅ graph_data 준비 완료: Data(x=[2076999, 38], edge_index=[2, 19060343], y=[2076999], train_mask=[2076999])


/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "



🚀 [실험 중] 현재 집계 방식: MEAN
  Epoch 01 | Loss: 3020881.3238
  Epoch 02 | Loss: 147778.8238
  Epoch 03 | Loss: 52443.2731
  Epoch 04 | Loss: 7426.7748
  Epoch 05 | Loss: 64.4073
  Epoch 06 | Loss: 59.4268
  Epoch 07 | Loss: 35.1431
  Epoch 08 | Loss: 24.8545
  Epoch 09 | Loss: 31.8413
  Epoch 10 | Loss: 16.7448
✅ MEAN 학습 완료! (소요 시간: 368.09초)

🚀 [실험 중] 현재 집계 방식: MAX
  Epoch 01 | Loss: 5794940.2670
  Epoch 02 | Loss: 417314.6563
  Epoch 03 | Loss: 17070.6719
  Epoch 04 | Loss: 6461.7388
  Epoch 05 | Loss: 3567.2591
  Epoch 06 | Loss: 8448.8700
  Epoch 07 | Loss: 1766.2762
  Epoch 08 | Loss: 75.7414
  Epoch 09 | Loss: 114.5463
  Epoch 10 | Loss: 32744.3642
✅ MAX 학습 완료! (소요 시간: 362.47초)

🚀 [실험 중] 현재 집계 방식: LSTM
  Epoch 01 | Loss: 215968.0761
  Epoch 02 | Loss: 2356.5517
  Epoch 03 | Loss: 115.5660
  Epoch 04 | Loss: 171.7008
  Epoch 05 | Loss: 64.0868
  Epoch 06 | Loss: 29.3300
  Epoch 07 | Loss: 25.5984
  Epoch 08 | Loss: 12.7112
  Epoch 09 | Loss: 8.7983
  Epoch 10 | Loss: 5.9351
✅ LSTM 학습 완

In [4]:
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score
import xgboost as xgb

def evaluate_strategy(model, aggr_name):
    print(f"🧐 [{aggr_name}] 최종 성능 평가 중...")
    model.eval()
    
    # 1. GNN 임베딩 추출 (Test 셋 포함 전체)
    full_loader = NeighborLoader(graph_data, num_neighbors=[15, 10], batch_size=2048, shuffle=False)
    all_emb = []
    with torch.no_grad():
        for batch in tqdm(full_loader, desc=f"Extrating {aggr_name}"):
            emb = model.get_embedding(batch.to(device).x, batch.edge_index)
            all_emb.append(emb[:batch.batch_size].cpu())
    
    final_emb = torch.cat(all_emb, dim=0).numpy()
    emb_cols = [f"gnn_emb_{i}" for i in range(64)]
    df_gnn = pl.concat([all_accounts.select("account_id"), pl.DataFrame(final_emb, schema=emb_cols)], how="horizontal")

    # 2. XGBoost 결합 및 데이터 분할
    df_hybrid = df_v2.join(df_gnn, on="account_id", how="left").fill_null(0.0)
    
    xgb_feat_cols = [c for c in df_hybrid.columns if c not in exclude_cols + ["date", "Timestamp"]]
    
    # OOT 분할 (Val/Test)
    val_cutoff = df_v2["time_group"][int(len(df_v2) * 0.8)]
    df_train = df_hybrid.filter(pl.col("time_group") < train_cutoff)
    df_val = df_hybrid.filter((pl.col("time_group") >= train_cutoff) & (pl.col("time_group") < val_cutoff))
    df_test = df_hybrid.filter(pl.col("time_group") >= val_cutoff)

    # 3. XGBoost 학습
    xgb_model = xgb.XGBClassifier(n_estimators=100, tree_method="hist", device="cuda", scale_pos_weight=48)
    xgb_model.fit(df_train.select(xgb_feat_cols).to_pandas(), df_train["is_laundering"].to_numpy(),
                  eval_set=[(df_val.select(xgb_feat_cols).to_pandas(), df_val["is_laundering"].to_numpy())], 
                  verbose=False, early_stopping_rounds=10)

    # 4. 테스트 결과 산출
    y_prob = xgb_model.predict_proba(df_test.select(xgb_feat_cols).to_pandas())[:, 1]
    y_test = df_test["is_laundering"].to_numpy()
    
    # 임계값은 Top-K 정밀도를 위해 동적으로 설정 (보통 확률 상위 0.1~0.5% 지점)
    threshold = np.percentile(y_prob, 99.5) 
    y_pred = (y_prob >= threshold).astype(int)

    return {
        "AUPRC": average_precision_score(y_test, y_prob),
        "F1": f1_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred)
    }

# 실행 예시 (위 루프가 끝난 후 model들이 살아있을 때)
# stats = evaluate_strategy(model, aggr)

In [1]:
import polars as pl
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.loader import NeighborLoader
import xgboost as xgb
import numpy as np
import pandas as pd
import time
import gc
from tqdm import tqdm
from sklearn.metrics import average_precision_score, f1_score, recall_score, precision_score

# 0. 경로 및 환경 설정
raw_path = "/home/tracerofjageum/HI-Medium_Master.parquet"
v2_path = "/home/tracerofjageum/aml_features_v2_integrated_final.parquet"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---------------------------------------------------------
# 1. 데이터 로드 및 graph_data 객체 생성
# ---------------------------------------------------------
print("📂 [1/4] 기초 데이터 로딩 및 그래프 구축 시작...")
df_raw = pl.read_parquet(raw_path)
df_v2 = pl.read_parquet(v2_path).with_columns(pl.col("account_id").cast(pl.Utf8))

# 시간 기준 설정 (6:2:2 분할)
df_v2 = df_v2.sort("time_group")
train_idx = int(len(df_v2) * 0.6)
train_cutoff = df_v2["time_group"][train_idx]

# 노드 및 엣지 생성 (시간 누수 차단)
all_accounts = pl.concat([
    df_raw.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id")),
    df_raw.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
]).unique().with_row_index("node_id")

df_raw_train = df_raw.filter(pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False) < train_cutoff)
df_edges = df_raw_train.select(["from_acc", "to_acc"]).with_columns([pl.col("from_acc").cast(pl.Utf8), pl.col("to_acc").cast(pl.Utf8)])
df_edges = df_edges.join(all_accounts.rename({"account_id": "from_acc", "node_id": "src_id"}), on="from_acc", how="left")
df_edges = df_edges.join(all_accounts.rename({"account_id": "to_acc", "node_id": "dst_id"}), on="to_acc", how="left")
edge_index = torch.tensor(df_edges.select(["src_id", "dst_id"]).to_numpy().T, dtype=torch.long)

# 노드 피처 및 라벨
exclude_cols = ["account_id", "time_group", "is_laundering", "mode_format", "currency_mode", "date", "Timestamp"]
gnn_feature_cols = [c for c in df_v2.columns if c not in exclude_cols]
df_v2_train = df_v2.filter(pl.col("time_group") < train_cutoff)
df_v2_node = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feature_cols])
df_node_features = all_accounts.join(df_v2_node, on="account_id", how="left").fill_null(0.0)

X = torch.tensor(df_node_features.select(gnn_feature_cols).to_numpy(), dtype=torch.float32)
target_labels = df_v2_train.filter(pl.col("is_laundering") == 1).select("account_id").unique().with_columns(pl.lit(1).alias("label"))
Y = torch.tensor(all_accounts.join(target_labels, on="account_id", how="left").fill_null(0)["label"].to_numpy(), dtype=torch.long)
train_mask = torch.tensor(all_accounts.join(pl.DataFrame({"account_id": df_v2_train["account_id"].unique(), "is_active": True}), on="account_id", how="left").fill_null(False)["is_active"].to_numpy(), dtype=torch.bool)

graph_data = Data(x=X, edge_index=edge_index, y=Y, train_mask=train_mask)
train_loader = NeighborLoader(graph_data, num_neighbors=[15, 10], batch_size=2048, input_nodes=graph_data.train_mask, shuffle=True)

# ---------------------------------------------------------
# 2. 모델 클래스 및 평가 함수 정의
# ---------------------------------------------------------
class GraphSAGE_Universal_Exp(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, aggr_type='mean'):
        super(GraphSAGE_Universal_Exp, self).__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels, aggr=aggr_type)
        self.conv2 = SAGEConv(hidden_channels, 1, aggr=aggr_type)

    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=0.3, training=self.training)
        return self.conv2(h, edge_index)
    
    @torch.no_grad()
    def get_embedding(self, x, edge_index):
        return F.relu(self.conv1(x, edge_index))

def evaluate_gnn_performance(model, aggr_name, graph_data, df_v2, all_accounts, train_cutoff):
    model.eval()
    full_loader = NeighborLoader(graph_data, num_neighbors=[15, 10], batch_size=2048, shuffle=False)
    all_emb = []
    with torch.no_grad():
        for batch in tqdm(full_loader, desc=f"🔮 {aggr_name} Embedding 추출"):
            batch = batch.to(device)
            emb = model.get_embedding(batch.x, batch.edge_index)
            all_emb.append(emb[:batch.batch_size].cpu())
    
    final_emb = torch.cat(all_emb, dim=0).numpy()
    emb_cols = [f"gnn_emb_{i}" for i in range(64)]
    df_gnn = pl.concat([all_accounts.select("account_id"), pl.DataFrame(final_emb, schema=emb_cols)], how="horizontal")

    df_hybrid = df_v2.join(df_gnn, on="account_id", how="left").fill_null(0.0)
    val_cutoff = df_v2["time_group"][int(len(df_v2) * 0.8)]
    xgb_feats = [c for c in df_hybrid.columns if c not in exclude_cols]

    df_train = df_hybrid.filter(pl.col("time_group") < train_cutoff)
    df_val   = df_hybrid.filter((pl.col("time_group") >= train_cutoff) & (pl.col("time_group") < val_cutoff))
    df_test  = df_hybrid.filter(pl.col("time_group") >= val_cutoff)

    # XGBoost 최신 버전 API 대응 (early_stopping_rounds 위치 수정)
    clf = xgb.XGBClassifier(
        n_estimators=100, max_depth=6, tree_method="hist", device="cuda", 
        scale_pos_weight=48, early_stopping_rounds=10, eval_metric="aucpr"
    )
    clf.fit(
        df_train.select(xgb_feats).to_pandas(), df_train["is_laundering"].to_numpy(),
        eval_set=[(df_val.select(xgb_feats).to_pandas(), df_val["is_laundering"].to_numpy())],
        verbose=0
    )

    y_prob = clf.predict_proba(df_test.select(xgb_feats).to_pandas())[:, 1]
    y_true = df_test["is_laundering"].to_numpy()
    threshold = np.percentile(y_prob, 99.5)
    y_pred = (y_prob >= threshold).astype(int)

    return {
        "Strategy": aggr_name,
        "AUPRC": average_precision_score(y_true, y_prob),
        "F1-Score": f1_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Precision": precision_score(y_true, y_pred, zero_division=0)
    }

# ---------------------------------------------------------
# 3. 통합 실험 루프 실행
# ---------------------------------------------------------
final_reports = []
aggr_list = ['mean', 'max', 'lstm']

for aggr in aggr_list:
    print(f"\n" + "="*50)
    print(f"🚀 [실험 시작] Aggregator: {aggr.upper()}")
    print("="*50)
    
    torch.manual_seed(42)
    start_time = time.time()
    
    model = GraphSAGE_Universal_Exp(graph_data.num_features, 64, aggr_type=aggr).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([48.0], device=device))
    
    model.train()
    for epoch in range(1, 11):
        total_loss = 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index).squeeze(-1)
            loss = criterion(out[:batch.batch_size], batch.y[:batch.batch_size].float())
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"  Epoch {epoch:02d} | Loss: {total_loss/len(train_loader):.4f}")
            
    report = evaluate_gnn_performance(model, aggr, graph_data, df_v2, all_accounts, train_cutoff)
    report["Time(sec)"] = time.time() - start_time
    final_reports.append(report)
    
    del model, optimizer; torch.cuda.empty_cache(); gc.collect()

# ---------------------------------------------------------
# 4. 최종 결과 리포트 출력
# ---------------------------------------------------------
df_results = pd.DataFrame(final_reports)
print("\n" + "🏆"*10 + " [최종 실험 결과 비교] " + "🏆"*10)
print(df_results.sort_values("AUPRC", ascending=False))

📂 [1/4] 기초 데이터 로딩 및 그래프 구축 시작...


/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "



🚀 [실험 시작] Aggregator: MEAN
  Epoch 01 | Loss: 3048969.9052
  Epoch 02 | Loss: 6554.8131
  Epoch 03 | Loss: 1426.4861
  Epoch 04 | Loss: 79.9395
  Epoch 05 | Loss: 97.9779
  Epoch 06 | Loss: 23.3359
  Epoch 07 | Loss: 29.8948
  Epoch 08 | Loss: 94.6425
  Epoch 09 | Loss: 45.1297
  Epoch 10 | Loss: 76.3048


🔮 mean Embedding 추출: 100%|███████████████████████████████████████████████| 1015/1015 [00:52<00:00, 19.21it/s]
/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/xgboost/core.py:751: UserWarning: [05:46:21] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)



🚀 [실험 시작] Aggregator: MAX
  Epoch 01 | Loss: 9754290.5877
  Epoch 02 | Loss: 1073049.7636
  Epoch 03 | Loss: 1134373.6601
  Epoch 04 | Loss: 145483.6469
  Epoch 05 | Loss: 62521.0353
  Epoch 06 | Loss: 102170.7183
  Epoch 07 | Loss: 7578.4837
  Epoch 08 | Loss: 1432.0505
  Epoch 09 | Loss: 226.4900
  Epoch 10 | Loss: 6956.2485


/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "
🔮 max Embedding 추출: 100%|████████████████████████████████████████████████| 1015/1015 [00:33<00:00, 30.68it/s]



🚀 [실험 시작] Aggregator: LSTM
  Epoch 01 | Loss: 697262.7884
  Epoch 02 | Loss: 73474.1721
  Epoch 03 | Loss: 5599.0965
  Epoch 04 | Loss: 15877.9812
  Epoch 05 | Loss: 4961.8982
  Epoch 06 | Loss: 16051.5533
  Epoch 07 | Loss: 685.1620
  Epoch 08 | Loss: 53.7333
  Epoch 09 | Loss: 32.5579
  Epoch 10 | Loss: 408.3696


/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "
🔮 lstm Embedding 추출: 100%|███████████████████████████████████████████████| 1015/1015 [00:38<00:00, 26.04it/s]



🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆 [최종 실험 결과 비교] 🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆
  Strategy     AUPRC  F1-Score    Recall  Precision   Time(sec)
1      max  0.533439  0.481522  0.578288   0.412498  508.068515
0     mean  0.531365  0.478963  0.575238   0.410294  722.669347
2     lstm  0.508542  0.467390  0.561735   0.400179  666.128411


In [2]:
import polars as pl
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.loader import NeighborLoader
import xgboost as xgb
import numpy as np
import pandas as pd
import time
import gc
from tqdm import tqdm
from sklearn.metrics import average_precision_score, f1_score, recall_score, precision_score

# 1. 하이브리드 전용 모델 클래스 정의
class GraphSAGE_Hybrid_Exp(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, aggr_l1, aggr_l2):
        super(GraphSAGE_Hybrid_Exp, self).__init__()
        # 1층과 2층의 집계 방식을 다르게 설정합니다.
        self.conv1 = SAGEConv(in_channels, hidden_channels, aggr=aggr_l1)
        self.conv2 = SAGEConv(hidden_channels, 1, aggr=aggr_l2)

    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=0.3, training=self.training)
        return self.conv2(h, edge_index)
    
    @torch.no_grad()
    def get_embedding(self, x, edge_index):
        # 1층의 결과물(64차원)을 추출합니다.
        return F.relu(self.conv1(x, edge_index))

# ---------------------------------------------------------
# 2. 하이브리드 실험 루프 실행
# ---------------------------------------------------------
hybrid_list = [('mean', 'max'), ('max', 'mean')]
hybrid_reports = []

for l1_type, l2_type in hybrid_list:
    strategy_name = f"{l1_type.upper()}_{l2_type.upper()}"
    print(f"\n" + "="*50)
    print(f"🚀 [하이브리드 실험] 1층: {l1_type.upper()} | 2층: {l2_type.upper()}")
    print("="*50)
    
    torch.manual_seed(42) # 공정한 비교를 위해 시드 고정
    start_time = time.time()
    
    # 모델 초기화 (동일한 64 히든 차원)
    model = GraphSAGE_Hybrid_Exp(graph_data.num_features, 64, l1_type, l2_type).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([48.0], device=device))
    
    # GNN 학습 (동일하게 10 Epoch)
    model.train()
    for epoch in range(1, 11):
        total_loss = 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index).squeeze(-1)
            loss = criterion(out[:batch.batch_size], batch.y[:batch.batch_size].float())
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"  Epoch {epoch:02d} | Loss: {total_loss/len(train_loader):.4f}")
            
    # 정밀 평가 수행 (앞서 정의한 evaluate_gnn_performance 함수 사용)
    report = evaluate_gnn_performance(model, strategy_name, graph_data, df_v2, all_accounts, train_cutoff)
    report["Time(sec)"] = time.time() - start_time
    hybrid_reports.append(report)
    
    del model, optimizer; torch.cuda.empty_cache(); gc.collect()

# ---------------------------------------------------------
# 3. 하이브리드 결과 비교 출력
# ---------------------------------------------------------
df_hybrid_results = pd.DataFrame(hybrid_reports)
print("\n" + "💎"*10 + " [하이브리드 실험 최종 결과] " + "💎"*10)
print(df_hybrid_results.sort_values("AUPRC", ascending=False))


🚀 [하이브리드 실험] 1층: MEAN | 2층: MAX
  Epoch 01 | Loss: 6331990.4319
  Epoch 02 | Loss: 982053.6247
  Epoch 03 | Loss: 1206253.4410
  Epoch 04 | Loss: 196591.6584
  Epoch 05 | Loss: 94934.4077
  Epoch 06 | Loss: 518537.2625
  Epoch 07 | Loss: 104362.3194
  Epoch 08 | Loss: 172926.7403
  Epoch 09 | Loss: 9041.6669
  Epoch 10 | Loss: 2235.6957


/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "
🔮 MEAN_MAX Embedding 추출: 100%|███████████████████████████████████████████| 1015/1015 [00:37<00:00, 27.32it/s]



🚀 [하이브리드 실험] 1층: MAX | 2층: MEAN
  Epoch 01 | Loss: 4313884.3475
  Epoch 02 | Loss: 3542.2881
  Epoch 03 | Loss: 16508.3056
  Epoch 04 | Loss: 834.6218
  Epoch 05 | Loss: 45.7814
  Epoch 06 | Loss: 35.9231
  Epoch 07 | Loss: 41.1217
  Epoch 08 | Loss: 123.5754
  Epoch 09 | Loss: 40.6716
  Epoch 10 | Loss: 23.1451


/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "
🔮 MAX_MEAN Embedding 추출: 100%|███████████████████████████████████████████| 1015/1015 [00:35<00:00, 28.38it/s]



💎💎💎💎💎💎💎💎💎💎 [하이브리드 실험 최종 결과] 💎💎💎💎💎💎💎💎💎💎
   Strategy     AUPRC  F1-Score    Recall  Precision   Time(sec)
0  MEAN_MAX  0.529175   0.47459  0.569963   0.406560  512.108727
1  MAX_MEAN  0.514247   0.46764  0.561831   0.400497  509.764094


# '잔차 연결' 레이어 추가

In [3]:
import polars as pl
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.loader import NeighborLoader
import xgboost as xgb
import numpy as np
import pandas as pd
import time
import gc
from tqdm import tqdm
from sklearn.metrics import average_precision_score, f1_score, recall_score, precision_score

# 0. 경로 및 환경 설정
raw_path = "/home/tracerofjageum/HI-Medium_Master.parquet"
v2_path = "/home/tracerofjageum/aml_features_v2_integrated_final.parquet"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---------------------------------------------------------
# 1. 데이터 로드 및 graph_data 객체 생성 (시간 누수 완벽 차단)
# ---------------------------------------------------------
print("📂 [1/4] 데이터 로딩 및 6:2:2 OOT 분할 시작...")
df_raw = pl.read_parquet(raw_path)
df_v2 = pl.read_parquet(v2_path).with_columns(pl.col("account_id").cast(pl.Utf8))

# 시간순 정렬 및 60% 분할점 계산
df_v2 = df_v2.sort("time_group")
train_cutoff = df_v2["time_group"][int(len(df_v2) * 0.6)]

# 전역 계좌 매핑
all_accounts = pl.concat([
    df_raw.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id")),
    df_raw.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
]).unique().with_row_index("node_id")

# [핵심] 훈련 기간 거래로만 엣지(인맥) 구축
df_raw_train = df_raw.filter(pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False) < train_cutoff)
df_edges = df_raw_train.select(["from_acc", "to_acc"]).with_columns([pl.col("from_acc").cast(pl.Utf8), pl.col("to_acc").cast(pl.Utf8)])
df_edges = df_edges.join(all_accounts.rename({"account_id": "from_acc", "node_id": "src_id"}), on="from_acc", how="left")
df_edges = df_edges.join(all_accounts.rename({"account_id": "to_acc", "node_id": "dst_id"}), on="to_acc", how="left")
edge_index = torch.tensor(df_edges.select(["src_id", "dst_id"]).to_numpy().T, dtype=torch.long)

# 노드 피처 및 라벨 생성 (9/22 이전 데이터만 평균)
exclude_cols = ["account_id", "time_group", "is_laundering", "mode_format", "currency_mode", "date", "Timestamp"]
gnn_feat_cols = [c for c in df_v2.columns if c not in exclude_cols]
df_v2_train = df_v2.filter(pl.col("time_group") < train_cutoff)
df_v2_node = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feat_cols])
df_node_feats = all_accounts.join(df_v2_node, on="account_id", how="left").fill_null(0.0)

X = torch.tensor(df_node_feats.select(gnn_feat_cols).to_numpy(), dtype=torch.float32)
target_labels = df_v2_train.filter(pl.col("is_laundering") == 1).select("account_id").unique().with_columns(pl.lit(1).alias("label"))
Y = torch.tensor(all_accounts.join(target_labels, on="account_id", how="left").fill_null(0)["label"].to_numpy(), dtype=torch.long)
mask = torch.tensor(all_accounts.join(pl.DataFrame({"account_id": df_v2_train["account_id"].unique(), "is_active": True}), on="account_id", how="left").fill_null(False)["is_active"].to_numpy(), dtype=torch.bool)

graph_data = Data(x=X, edge_index=edge_index, y=Y, train_mask=mask)
train_loader = NeighborLoader(graph_data, num_neighbors=[15, 10], batch_size=2048, input_nodes=graph_data.train_mask, shuffle=True)

# ---------------------------------------------------------
# 2. 잔차 연결 기반 MAX 모델 클래스 정의
# ---------------------------------------------------------
class GraphSAGE_Residual_MAX(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(GraphSAGE_Residual_MAX, self).__init__()
        # 챔피언 설정: aggr='max' (Pooling)
        self.conv1 = SAGEConv(in_channels, hidden_channels, aggr='max')
        self.conv2 = SAGEConv(hidden_channels, 1, aggr='max')
        
        # [Residual] 입력 피처와 히든 차원을 맞추는 선형 레이어
        self.res_lin = torch.nn.Linear(in_channels, hidden_channels)

    def forward(self, x, edge_index):
        # 1층: 인맥 요약(MAX) + 내 행동 유지(Residual)
        h = self.conv1(x, edge_index)
        res = self.res_lin(x)
        h = F.relu(h + res) # 여기서 '인맥'과 '내 정보'가 합쳐집니다.
        
        h = F.dropout(h, p=0.3, training=self.training)
        # 2층
        return self.conv2(h, edge_index)
    
    @torch.no_grad()
    def get_embedding(self, x, edge_index):
        # 임베딩 추출 시에도 잔차 연결을 포함하여 정보 손실 방지
        h = self.conv1(x, edge_index)
        res = self.res_lin(x)
        return F.relu(h + res)

# ---------------------------------------------------------
# 3. 성능 평가 및 실험 실행
# ---------------------------------------------------------
def run_final_experiment(graph_data, df_v2, all_accounts, train_cutoff):
    print(f"\n🚀 [챔피언 도전] MAX Aggregator + 잔차 연결(Residual) 실험 시작")
    print("="*60)
    
    torch.manual_seed(42)
    start_time = time.time()
    
    # 모델 학습
    model = GraphSAGE_Residual_MAX(graph_data.num_features, 64).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([48.0], device=device))
    
    model.train()
    for epoch in range(1, 11):
        total_loss = 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index).squeeze(-1)
            loss = criterion(out[:batch.batch_size], batch.y[:batch.batch_size].float())
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"  Epoch {epoch:02d} | Loss: {total_loss/len(train_loader):.4f}")

    # 평가: 임베딩 추출
    model.eval()
    full_loader = NeighborLoader(graph_data, num_neighbors=[15, 10], batch_size=2048, shuffle=False)
    all_emb = []
    with torch.no_grad():
        for batch in tqdm(full_loader, desc="🔮 Embedding 추출"):
            batch = batch.to(device)
            emb = model.get_embedding(batch.x, batch.edge_index)
            all_emb.append(emb[:batch.batch_size].cpu())
    
    # XGBoost 검증 (OOT 분리)
    final_emb = torch.cat(all_emb, dim=0).numpy()
    df_gnn = pl.concat([all_accounts.select("account_id"), pl.DataFrame(final_emb, schema=[f"gnn_emb_{i}" for i in range(64)])], how="horizontal")
    df_hybrid = df_v2.join(df_gnn, on="account_id", how="left").fill_null(0.0)
    
    val_cutoff = df_v2["time_group"][int(len(df_v2) * 0.8)]
    xgb_feats = [c for c in df_hybrid.columns if c not in exclude_cols]

    df_train = df_hybrid.filter(pl.col("time_group") < train_cutoff)
    df_val   = df_hybrid.filter((pl.col("time_group") >= train_cutoff) & (pl.col("time_group") < val_cutoff))
    df_test  = df_hybrid.filter(pl.col("time_group") >= val_cutoff)

    # 최신 XGBoost API 적용
    clf = xgb.XGBClassifier(
        n_estimators=100, max_depth=6, tree_method="hist", device="cuda", 
        scale_pos_weight=48, early_stopping_rounds=10, eval_metric="aucpr"
    )
    clf.fit(
        df_train.select(xgb_feats).to_pandas(), df_train["is_laundering"].to_numpy(),
        eval_set=[(df_val.select(xgb_feats).to_pandas(), df_val["is_laundering"].to_numpy())],
        verbose=0
    )

    # 결과 리포트
    y_prob = clf.predict_proba(df_test.select(xgb_feats).to_pandas())[:, 1]
    y_true = df_test["is_laundering"].to_numpy()
    y_pred = (y_prob >= np.percentile(y_prob, 99.5)).astype(int)

    print("\n" + "🏆"*10 + " 최종 결과 " + "🏆"*10)
    print(f" - AUPRC    : {average_precision_score(y_true, y_prob):.4f} (이전 최고: 0.5334)")
    print(f" - Recall   : {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f" - Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f" - 소요 시간: {time.time() - start_time:.2f} sec")

# 실행
run_final_experiment(graph_data, df_v2, all_accounts, train_cutoff)

📂 [1/4] 데이터 로딩 및 6:2:2 OOT 분할 시작...


/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "



🚀 [챔피언 도전] MAX Aggregator + 잔차 연결(Residual) 실험 시작
  Epoch 01 | Loss: 6771665.4478
  Epoch 02 | Loss: 6131.8074
  Epoch 03 | Loss: 307.5501
  Epoch 04 | Loss: 183.7593
  Epoch 05 | Loss: 1057.0496
  Epoch 06 | Loss: 63.8080
  Epoch 07 | Loss: 59.2966
  Epoch 08 | Loss: 61.0180
  Epoch 09 | Loss: 45.9429
  Epoch 10 | Loss: 33.8710


🔮 Embedding 추출: 100%|████████████████████████████████████████████████████| 1015/1015 [00:36<00:00, 28.09it/s]



🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆 최종 결과 🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆
 - AUPRC    : 0.5270 (이전 최고: 0.5334)
 - Recall   : 0.5696
 - Precision: 0.4063
 - 소요 시간: 515.63 sec


# 멀티집계(mean+max)

리스트 형태로 집계 함수를 넘기면, 내부적으로 두 가지 안경을 동시에 끼고 정보를 수집한 뒤 하나로 길게 이어 붙여(Concatenate) 줍니다.  
즉, XGBoost가 "주변 평균은 이렇고, 그중 튀는 값은 저렇구나!*를 동시에 알 수 있게 되는 엄청난 장점

In [4]:
import polars as pl
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.loader import NeighborLoader
import xgboost as xgb
import numpy as np
import pandas as pd
import time
import gc
from tqdm import tqdm
from sklearn.metrics import average_precision_score, f1_score, recall_score, precision_score

# 0. 경로 및 환경 설정
raw_path = "/home/tracerofjageum/HI-Medium_Master.parquet"
v2_path = "/home/tracerofjageum/aml_features_v2_integrated_final.parquet"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---------------------------------------------------------
# 1. 데이터 세팅 (시간 누수 완벽 차단)
# ---------------------------------------------------------
print("📂 [1/4] 데이터 로딩 및 6:2:2 OOT 분할 시작...")
df_raw = pl.read_parquet(raw_path)
df_v2 = pl.read_parquet(v2_path).with_columns(pl.col("account_id").cast(pl.Utf8))

df_v2 = df_v2.sort("time_group")
train_cutoff = df_v2["time_group"][int(len(df_v2) * 0.6)]

all_accounts = pl.concat([
    df_raw.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id")),
    df_raw.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
]).unique().with_row_index("node_id")

df_raw_train = df_raw.filter(pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False) < train_cutoff)
df_edges = df_raw_train.select(["from_acc", "to_acc"]).with_columns([pl.col("from_acc").cast(pl.Utf8), pl.col("to_acc").cast(pl.Utf8)])
df_edges = df_edges.join(all_accounts.rename({"account_id": "from_acc", "node_id": "src_id"}), on="from_acc", how="left")
df_edges = df_edges.join(all_accounts.rename({"account_id": "to_acc", "node_id": "dst_id"}), on="to_acc", how="left")
edge_index = torch.tensor(df_edges.select(["src_id", "dst_id"]).to_numpy().T, dtype=torch.long)

exclude_cols = ["account_id", "time_group", "is_laundering", "mode_format", "currency_mode", "date", "Timestamp"]
gnn_feat_cols = [c for c in df_v2.columns if c not in exclude_cols]
df_v2_train = df_v2.filter(pl.col("time_group") < train_cutoff)
df_v2_node = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feat_cols])
df_node_feats = all_accounts.join(df_v2_node, on="account_id", how="left").fill_null(0.0)

X = torch.tensor(df_node_feats.select(gnn_feat_cols).to_numpy(), dtype=torch.float32)
target_labels = df_v2_train.filter(pl.col("is_laundering") == 1).select("account_id").unique().with_columns(pl.lit(1).alias("label"))
Y = torch.tensor(all_accounts.join(target_labels, on="account_id", how="left").fill_null(0)["label"].to_numpy(), dtype=torch.long)
mask = torch.tensor(all_accounts.join(pl.DataFrame({"account_id": df_v2_train["account_id"].unique(), "is_active": True}), on="account_id", how="left").fill_null(False)["is_active"].to_numpy(), dtype=torch.bool)

graph_data = Data(x=X, edge_index=edge_index, y=Y, train_mask=mask)
train_loader = NeighborLoader(graph_data, num_neighbors=[15, 10], batch_size=2048, input_nodes=graph_data.train_mask, shuffle=True)

# ---------------------------------------------------------
# 2. 비교할 두 가지 GNN 아키텍처 정의
# ---------------------------------------------------------
# [A] 챔피언 모델: 단일 MAX
class GraphSAGE_Single_MAX(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(GraphSAGE_Single_MAX, self).__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels, aggr='max')
        self.conv2 = SAGEConv(hidden_channels, 1, aggr='max')

    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.dropout(h, p=0.3, training=self.training)
        return self.conv2(h, edge_index)
    
    @torch.no_grad()
    def get_embedding(self, x, edge_index):
        return F.relu(self.conv1(x, edge_index))

# [B] 도전 모델: 멀티 집계 (MEAN + MAX 동시 적용)
class GraphSAGE_MultiAggr(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(GraphSAGE_MultiAggr, self).__init__()
        # PyG 최신 기능: aggr 파라미터에 리스트를 주면 자동으로 멀티 집계 후 Concat!
        aggr_list = ['mean', 'max']
        self.conv1 = SAGEConv(in_channels, hidden_channels, aggr=aggr_list)
        self.conv2 = SAGEConv(hidden_channels, 1, aggr=aggr_list)

    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.dropout(h, p=0.3, training=self.training)
        return self.conv2(h, edge_index)
    
    @torch.no_grad()
    def get_embedding(self, x, edge_index):
        # 멀티 집계된 풍부한 임베딩 벡터 반환
        return F.relu(self.conv1(x, edge_index))

# ---------------------------------------------------------
# 3. 평가 함수 (XGBoost 연동)
# ---------------------------------------------------------
def evaluate_model(model, model_name, graph_data, df_v2, all_accounts, train_cutoff):
    model.eval()
    full_loader = NeighborLoader(graph_data, num_neighbors=[15, 10], batch_size=2048, shuffle=False)
    all_emb = []
    with torch.no_grad():
        for batch in tqdm(full_loader, desc=f"🔮 {model_name} Embedding 추출"):
            batch = batch.to(device)
            emb = model.get_embedding(batch.x, batch.edge_index)
            all_emb.append(emb[:batch.batch_size].cpu())
    
    final_emb = torch.cat(all_emb, dim=0).numpy()
    
    # 멀티 집계 시 임베딩 차원이 늘어날 수 있으므로 동적으로 차원수 할당
    emb_dim = final_emb.shape[1] 
    emb_cols = [f"gnn_emb_{i}" for i in range(emb_dim)]
    df_gnn = pl.concat([all_accounts.select("account_id"), pl.DataFrame(final_emb, schema=emb_cols)], how="horizontal")

    df_hybrid = df_v2.join(df_gnn, on="account_id", how="left").fill_null(0.0)
    val_cutoff = df_v2["time_group"][int(len(df_v2) * 0.8)]
    xgb_feats = [c for c in df_hybrid.columns if c not in exclude_cols]

    df_train = df_hybrid.filter(pl.col("time_group") < train_cutoff)
    df_val   = df_hybrid.filter((pl.col("time_group") >= train_cutoff) & (pl.col("time_group") < val_cutoff))
    df_test  = df_hybrid.filter(pl.col("time_group") >= val_cutoff)

    clf = xgb.XGBClassifier(
        n_estimators=100, max_depth=6, tree_method="hist", device="cuda", 
        scale_pos_weight=48, early_stopping_rounds=10, eval_metric="aucpr"
    )
    clf.fit(
        df_train.select(xgb_feats).to_pandas(), df_train["is_laundering"].to_numpy(),
        eval_set=[(df_val.select(xgb_feats).to_pandas(), df_val["is_laundering"].to_numpy())],
        verbose=0
    )

    y_prob = clf.predict_proba(df_test.select(xgb_feats).to_pandas())[:, 1]
    y_true = df_test["is_laundering"].to_numpy()
    y_pred = (y_prob >= np.percentile(y_prob, 99.5)).astype(int)

    return {
        "Model": model_name,
        "AUPRC": average_precision_score(y_true, y_prob),
        "F1-Score": f1_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Precision": precision_score(y_true, y_pred, zero_division=0)
    }

# ---------------------------------------------------------
# 4. 1:1 진검승부 실행
# ---------------------------------------------------------
experiments = {
    "Single_MAX (기존 챔피언)": GraphSAGE_Single_MAX(graph_data.num_features, 64),
    "Multi_Aggr (MEAN+MAX)": GraphSAGE_MultiAggr(graph_data.num_features, 64)
}

results = []

for name, model in experiments.items():
    print(f"\n" + "="*50)
    print(f"🚀 실험 시작: {name}")
    print("="*50)
    
    torch.manual_seed(42)
    start_time = time.time()
    
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([48.0], device=device))
    
    # GNN 학습
    model.train()
    for epoch in range(1, 11):
        total_loss = 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index).squeeze(-1)
            loss = criterion(out[:batch.batch_size], batch.y[:batch.batch_size].float())
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"  Epoch {epoch:02d} | Loss: {total_loss/len(train_loader):.4f}")
            
    # XGBoost 평가
    report = evaluate_model(model, name, graph_data, df_v2, all_accounts, train_cutoff)
    report["Time(sec)"] = time.time() - start_time
    results.append(report)
    
    del model, optimizer; torch.cuda.empty_cache(); gc.collect()

# ---------------------------------------------------------
# 5. 최종 결과 리포트 출력
# ---------------------------------------------------------
df_results = pd.DataFrame(results)
print("\n" + "🏆"*10 + " [결과 비교 테이블] " + "🏆"*10)
print(df_results.sort_values("AUPRC", ascending=False))

📂 [1/4] 데이터 로딩 및 6:2:2 OOT 분할 시작...


/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "



🚀 실험 시작: Single_MAX (기존 챔피언)
  Epoch 01 | Loss: 4796409.3184
  Epoch 02 | Loss: 35215.3353
  Epoch 03 | Loss: 5520.6960
  Epoch 04 | Loss: 10485.6693
  Epoch 05 | Loss: 972.3923
  Epoch 06 | Loss: 511.3981
  Epoch 07 | Loss: 101.8757
  Epoch 08 | Loss: 59.9204
  Epoch 09 | Loss: 52.7004
  Epoch 10 | Loss: 37.2023


🔮 Single_MAX (기존 챔피언) Embedding 추출: 100%|███████████████████████████| 1015/1015 [00:33<00:00, 30.02it/s]



🚀 실험 시작: Multi_Aggr (MEAN+MAX)
  Epoch 01 | Loss: 5703812.9596
  Epoch 02 | Loss: 82071.3202
  Epoch 03 | Loss: 146717.5354
  Epoch 04 | Loss: 19747.2527
  Epoch 05 | Loss: 2502.3901
  Epoch 06 | Loss: 6933.8995
  Epoch 07 | Loss: 1300.9358
  Epoch 08 | Loss: 100.7811
  Epoch 09 | Loss: 1352.9925
  Epoch 10 | Loss: 27.9281


/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "
🔮 Multi_Aggr (MEAN+MAX) Embedding 추출: 100%|██████████████████████████████| 1015/1015 [00:35<00:00, 28.42it/s]



🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆 [결과 비교 테이블] 🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆
                   Model     AUPRC  F1-Score    Recall  Precision   Time(sec)
0    Single_MAX (기존 챔피언)  0.535210  0.476653  0.572625   0.408233  493.523203
1  Multi_Aggr (MEAN+MAX)  0.527289  0.472727  0.567736   0.404958  515.811963


# pna 실험

In [3]:
import polars as pl
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv, PNAConv
from torch_geometric.loader import NeighborLoader
from torch_geometric.utils import degree
import xgboost as xgb
import numpy as np
import pandas as pd
import time
import gc
from tqdm import tqdm
from sklearn.metrics import average_precision_score, f1_score, recall_score, precision_score

# ---------------------------------------------------------
# 1. [데이터 로드] PDF 제약 준수 (1시간+계좌 단위)
# ---------------------------------------------------------
print("📂 기초 데이터 로딩 및 OOT 분할 시작...")
raw_path = "/home/tracerofjageum/HI-Medium_Master.parquet"
v2_path = "/home/tracerofjageum/aml_features_v2_integrated_final.parquet"

df_raw = pl.read_parquet(raw_path)
df_v2 = pl.read_parquet(v2_path).with_columns(pl.col("account_id").cast(pl.Utf8))

# 시간순 정렬 및 6:2:2 OOT 분할 지점 계산 (시간 누수 차단)
df_v2 = df_v2.sort("time_group")
train_cutoff = df_v2["time_group"][int(len(df_v2) * 0.6)]
val_cutoff = df_v2["time_group"][int(len(df_v2) * 0.8)]

# ---------------------------------------------------------
# 2. [그래프 구축] 시간 누수(Leakage) 원천 차단 로직
# ---------------------------------------------------------
# 노드 매핑
all_accounts = pl.concat([
    df_raw.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id")),
    df_raw.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
]).unique().with_row_index("node_id")

# [핵심] 훈련 기간(Train Cutoff) 이전의 거래로만 엣지(인맥) 구성
df_raw_train = df_raw.filter(
    pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False) < train_cutoff
)

df_edges = df_raw_train.select(["from_acc", "to_acc"]).with_columns([pl.col("from_acc").cast(pl.Utf8), pl.col("to_acc").cast(pl.Utf8)])
df_edges = df_edges.join(all_accounts.rename({"account_id": "from_acc", "node_id": "src_id"}), on="from_acc", how="left")
df_edges = df_edges.join(all_accounts.rename({"account_id": "to_acc", "node_id": "dst_id"}), on="to_acc", how="left")
edge_index = torch.tensor(df_edges.select(["src_id", "dst_id"]).to_numpy().T, dtype=torch.long)

# ---------------------------------------------------------
# 3. [PNA 준비] Degree Histogram 계산
# ---------------------------------------------------------
# 훈련 노드들의 차수 분포를 계산하여 PNA 스케일러에 전달
deg = degree(edge_index[0], num_nodes=all_accounts.height)
deg_hist = torch.zeros(int(deg.max().item()) + 1)
deg_hist.scatter_add_(0, deg.to(torch.long), torch.ones_like(deg))

# ---------------------------------------------------------
# 4. [PNA 모델] PDF 로드맵: 급격한 자금 흐름 탐지 특화
# ---------------------------------------------------------
class PNA_AML_Model(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, deg_hist):
        super(PNA_AML_Model, self).__init__()
        # PDF 6p 지침 반영: Max/Std 집계를 통해 변동성 포착
        aggregators = ['mean', 'min', 'max', 'std']
        scalers = ['identity', 'amplification', 'attenuation']
        
        self.conv1 = PNAConv(in_channels, hidden_channels, aggregators=aggregators, scalers=scalers, deg=deg_hist)
        self.conv2 = PNAConv(hidden_channels, 1, aggregators=aggregators, scalers=scalers, deg=deg_hist)

    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=0.3, training=self.training)
        return self.conv2(h, edge_index)

    @torch.no_grad()
    def get_embedding(self, x, edge_index):
        return F.relu(self.conv1(x, edge_index))

# ---------------------------------------------------------
# 5. 실행 및 평가 (XGBoost 하이브리드 결합)
# ---------------------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# (이후 노드 피처 생성 및 graph_data 객체 생성 과정은 이전과 동일하게 수행)
print("🚀 PNA 실험 모델 학습 및 성능 평가를 시작합니다...")

📂 기초 데이터 로딩 및 OOT 분할 시작...
🚀 PNA 실험 모델 학습 및 성능 평가를 시작합니다...


In [4]:
import polars as pl
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import PNAConv
from torch_geometric.loader import NeighborLoader
from torch_geometric.utils import degree
import xgboost as xgb
import numpy as np
import pandas as pd
import time
import gc
from tqdm import tqdm
from sklearn.metrics import average_precision_score, f1_score, recall_score, precision_score

# 0. 경로 및 장치 설정
RAW_PATH = "/home/tracerofjageum/HI-Medium_Master.parquet"
V2_PATH = "/home/tracerofjageum/aml_features_v2_integrated_final.parquet"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---------------------------------------------------------
# 1. [데이터 로드] PDF 제약 준수 (1시간+계좌 단위)
# ---------------------------------------------------------
print("📂 [1/5] 데이터 로딩 및 6:2:2 OOT 분할 시작...")
df_raw = pl.read_parquet(RAW_PATH)
df_v2 = pl.read_parquet(V2_PATH).with_columns(pl.col("account_id").cast(pl.Utf8))

# 시간순 정렬 및 OOT 분할 지점 계산 (시간 누수 방지)
df_v2 = df_v2.sort("time_group")
total_len = len(df_v2)
train_cutoff = df_v2["time_group"][int(total_len * 0.6)]
val_cutoff = df_v2["time_group"][int(total_len * 0.8)]

# ---------------------------------------------------------
# 2. [그래프 구축] 시간 누수(Leakage) 원천 차단
# ---------------------------------------------------------
# 전역 계좌 매핑
all_accounts = pl.concat([
    df_raw.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id")),
    df_raw.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
]).unique().with_row_index("node_id")

# [핵심] 훈련 기간(Train Cutoff) 이전의 거래로만 인맥(Edge) 구축
df_raw_train = df_raw.filter(
    pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False) < train_cutoff
)

df_edges = df_raw_train.select(["from_acc", "to_acc"]).with_columns([
    pl.col("from_acc").cast(pl.Utf8), 
    pl.col("to_acc").cast(pl.Utf8)
])
df_edges = df_edges.join(all_accounts.rename({"account_id": "from_acc", "node_id": "src_id"}), on="from_acc", how="left")
df_edges = df_edges.join(all_accounts.rename({"account_id": "to_acc", "node_id": "dst_id"}), on="to_acc", how="left")
edge_index = torch.tensor(df_edges.select(["src_id", "dst_id"]).to_numpy().T, dtype=torch.long)

# ---------------------------------------------------------
# 3. [노드 피처 & 라벨] 1시간 단위 계좌 기반 생성
# ---------------------------------------------------------
exclude_cols = ["account_id", "time_group", "is_laundering", "mode_format", "currency_mode", "date", "Timestamp", "timestamp"]
gnn_feat_cols = [c for c in df_v2.columns if c not in exclude_cols]

# 훈련 데이터 기반 노드 피처 (평균값 사용)
df_v2_train = df_v2.filter(pl.col("time_group") < train_cutoff)
df_v2_node = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feat_cols])
df_node_feats = all_accounts.join(df_v2_node, on="account_id", how="left").fill_null(0.0)

X = torch.tensor(df_node_feats.select(gnn_feat_cols).to_numpy(), dtype=torch.float32)
target_labels = df_v2_train.filter(pl.col("is_laundering") == 1).select("account_id").unique().with_columns(pl.lit(1).alias("label"))
Y = torch.tensor(all_accounts.join(target_labels, on="account_id", how="left").fill_null(0)["label"].to_numpy(), dtype=torch.long)
mask = torch.tensor(all_accounts.join(pl.DataFrame({"account_id": df_v2_train["account_id"].unique(), "is_active": True}), on="account_id", how="left").fill_null(False)["is_active"].to_numpy(), dtype=torch.bool)

graph_data = Data(x=X, edge_index=edge_index, y=Y, train_mask=mask)

# PNA를 위한 Degree Histogram 계산
deg = degree(edge_index[0], num_nodes=X.size(0))
deg_hist = torch.zeros(int(deg.max().item()) + 1)
deg_hist.scatter_add_(0, deg.to(torch.long), torch.ones_like(deg))

# ---------------------------------------------------------
# 4. [PNA 모델 정의] PDF 로드맵 4주차 고도화 아키텍처
# ---------------------------------------------------------
class PNA_AML_Model(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, deg_hist):
        super(PNA_AML_Model, self).__init__()
        aggregators = ['mean', 'min', 'max', 'std']
        scalers = ['identity', 'amplification', 'attenuation']
        self.conv1 = PNAConv(in_channels, hidden_channels, aggregators=aggregators, scalers=scalers, deg=deg_hist)
        self.conv2 = PNAConv(hidden_channels, 1, aggregators=aggregators, scalers=scalers, deg=deg_hist)

    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.dropout(h, p=0.3, training=self.training)
        return self.conv2(h, edge_index)

    @torch.no_grad()
    def get_embedding(self, x, edge_index):
        return F.relu(self.conv1(x, edge_index))

# ---------------------------------------------------------
# 5. [학습 및 평가] XGBoost 하이브리드 결합
# ---------------------------------------------------------
print("🚀 [PNA 배틀] 모델 학습 및 성능 평가 시작...")
model = PNA_AML_Model(graph_data.num_features, 64, deg_hist).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([48.0], device=DEVICE))

train_loader = NeighborLoader(graph_data, num_neighbors=[15, 10], batch_size=2048, input_nodes=graph_data.train_mask, shuffle=True)

model.train()
for epoch in range(1, 11):
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index).squeeze(-1)
        loss = criterion(out[:batch.batch_size], batch.y[:batch.batch_size].float())
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"  Epoch {epoch:02d} | Loss: {total_loss/len(train_loader):.4f}")

# 임베딩 추출 및 XGBoost 결합 평가
model.eval()
full_loader = NeighborLoader(graph_data, num_neighbors=[15, 10], batch_size=2048, shuffle=False)
all_emb = []
with torch.no_grad():
    for batch in tqdm(full_loader, desc="🔮 PNA 임베딩 추출"):
        emb = model.get_embedding(batch.to(DEVICE).x, batch.edge_index)
        all_emb.append(emb[:batch.batch_size].cpu())

df_gnn = pl.concat([all_accounts.select("account_id"), pl.DataFrame(torch.cat(all_emb).numpy(), schema=[f"pna_emb_{i}" for i in range(64)])], how="horizontal")
df_hybrid = df_v2.join(df_gnn, on="account_id", how="left").fill_null(0.0)

# 최종 평가 (XGBoost)
xgb_feats = [c for c in df_hybrid.columns if c not in exclude_cols]
train_pd = df_hybrid.filter(pl.col("time_group") < train_cutoff).select(xgb_feats + ["is_laundering"]).to_pandas()
test_pd = df_hybrid.filter(pl.col("time_group") >= val_cutoff).select(xgb_feats + ["is_laundering"]).to_pandas()

clf = xgb.XGBClassifier(n_estimators=100, tree_method="hist", device="cuda", scale_pos_weight=48)
clf.fit(train_pd[xgb_feats], train_pd["is_laundering"])

y_prob = clf.predict_proba(test_pd[xgb_feats])[:, 1]
print(f"\n🏆 [PNA 앙상블 성적] AUPRC: {average_precision_score(test_pd['is_laundering'], y_prob):.4f}")

📂 [1/5] 데이터 로딩 및 6:2:2 OOT 분할 시작...
🚀 [PNA 배틀] 모델 학습 및 성능 평가 시작...


/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "


  Epoch 01 | Loss: 177693.1235
  Epoch 02 | Loss: 7.3803
  Epoch 03 | Loss: 6.5745
  Epoch 04 | Loss: 3.2225
  Epoch 05 | Loss: 1.9089
  Epoch 06 | Loss: 3.7960
  Epoch 07 | Loss: 7.3911
  Epoch 08 | Loss: 1.9013
  Epoch 09 | Loss: 4.9563
  Epoch 10 | Loss: 11.5628


🔮 PNA 임베딩 추출: 100%|███████████████████████████████████████████████████| 1015/1015 [00:49<00:00, 20.62it/s]
/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/xgboost/core.py:751: UserWarning: [07:27:13] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)



🏆 [PNA 앙상블 성적] AUPRC: 0.5350


In [6]:
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
import pandas as pd

# ---------------------------------------------------------
# 1. [오류 해결] test_full 정의 및 y_true 추출
# ---------------------------------------------------------
# XGBoost 학습 시 사용했던 val_cutoff와 동일한 기준으로 Polars 데이터프레임 생성
test_full = df_hybrid.filter(pl.col("time_group") >= val_cutoff)
y_true = test_pd['is_laundering'].values

# ---------------------------------------------------------
# 2. [평가 함수] 상세 지표 및 일별 Top-100 분석
# ---------------------------------------------------------
def print_pna_detailed_report(y_true, y_prob, test_df_pl, threshold=0.7):
    # (1) F1, Precision, Recall (임계값 0.7 기준)
    y_pred = (y_prob > threshold).astype(int)
    print("\n" + "="*70)
    print(f"🏆 [PNA 앙상블] 성능 평가 리포트 (Threshold: {threshold})")
    print("="*70)
    print(classification_report(y_true, y_pred, digits=4))
    
    # 
    
    # (2) 전체 Top-K 탐지 성능 (100, 500, 1000, 5000)
    eval_df = pd.DataFrame({'prob': y_prob, 'actual': y_true})
    sorted_df = eval_df.sort_values(by='prob', ascending=False).reset_index(drop=True)
    
    print(f"\n[전체 구간 Top-K 검거 성능]")
    print("-" * 55)
    for k in [100, 500, 1000, 5000]:
        if len(sorted_df) < k: continue
        hits = int(sorted_df.head(k)['actual'].sum())
        prec = (hits / k) * 100
        # Recall 계산 (전체 사기꾼 중 k명 안에서 잡은 비율)
        total_frauds = y_true.sum()
        recall = (hits / total_frauds) * 100
        print(f"Top {k:<5} | 검거: {hits:<5} | 정밀도: {prec:>6.2f}% | 재현율: {recall:>6.2f}%")

    # 

    # (3) 테스트 데이터 일자별 Top-100 탐지 추이
    # Polars의 time_group을 활용해 날짜별로 그룹화
    eval_df['date'] = test_df_pl['time_group'].dt.date().to_pandas()
    daily_results = []
    
    for d in sorted(eval_df['date'].unique()):
        day_data = eval_df[eval_df['date'] == d]
        if len(day_data) < 100: continue
        day_hits = int(day_data.sort_values(by='prob', ascending=False).head(100)['actual'].sum())
        daily_results.append({'Date': d, 'Hits@100': day_hits, 'Precision': f"{day_hits}%"})
    
    print(f"\n[일자별 Top-100 탐지 추이]")
    print("-" * 55)
    print(pd.DataFrame(daily_results).to_string(index=False))

# ---------------------------------------------------------
# 3. 결과 출력
# ---------------------------------------------------------
print_pna_detailed_report(y_true, y_prob, test_full)


🏆 [PNA 앙상블] 성능 평가 리포트 (Threshold: 0.7)
              precision    recall  f1-score   support

       False     0.9988    0.9938    0.9963   5772252
        True     0.2820    0.6784    0.3984     20661

    accuracy                         0.9927   5792913
   macro avg     0.6404    0.8361    0.6974   5792913
weighted avg     0.9963    0.9927    0.9942   5792913


[전체 구간 Top-K 검거 성능]
-------------------------------------------------------
Top 100   | 검거: 100   | 정밀도: 100.00% | 재현율:   0.48%
Top 500   | 검거: 493   | 정밀도:  98.60% | 재현율:   2.39%
Top 1000  | 검거: 976   | 정밀도:  97.60% | 재현율:   4.72%
Top 5000  | 검거: 4585  | 정밀도:  91.70% | 재현율:  22.19%

[일자별 Top-100 탐지 추이]
-------------------------------------------------------
      Date  Hits@100 Precision
2022-09-14        95       95%
2022-09-15        95       95%
2022-09-16        86       86%
2022-09-17       100      100%
2022-09-18       100      100%
2022-09-19       100      100%
2022-09-20       100      100%
2022-09-21       100   

In [8]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, PNAConv
from torch_geometric.utils import degree
import pandas as pd
import numpy as np
import time
import gc

# ---------------------------------------------------------
# [필수] 클래스 정의: 1. 기존 챔피언 (Single MAX)
# ---------------------------------------------------------
class GraphSAGE_Single_MAX(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(GraphSAGE_Single_MAX, self).__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels, aggr='max')
        self.conv2 = SAGEConv(hidden_channels, 1, aggr='max')

    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.dropout(h, p=0.3, training=self.training)
        return self.conv2(h, edge_index)
    
    @torch.no_grad()
    def get_embedding(self, x, edge_index):
        return F.relu(self.conv1(x, edge_index))

# ---------------------------------------------------------
# [필수] 클래스 정의: 2. 고도화 모델 (PNA)
# ---------------------------------------------------------
class GraphSAGE_PNA(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, deg_hist):
        super(GraphSAGE_PNA, self).__init__()
        aggregators = ['mean', 'min', 'max', 'std']
        scalers = ['identity', 'amplification', 'attenuation']
        
        self.conv1 = PNAConv(in_channels, hidden_channels, 
                             aggregators=aggregators, scalers=scalers, deg=deg_hist)
        self.conv2 = PNAConv(hidden_channels, 1, 
                             aggregators=aggregators, scalers=scalers, deg=deg_hist)

    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.dropout(h, p=0.3, training=self.training)
        return self.conv2(h, edge_index)
    
    @torch.no_grad()
    def get_embedding(self, x, edge_index):
        return F.relu(self.conv1(x, edge_index))

# ---------------------------------------------------------
# 3. PNA 전용 Degree Histogram 계산 및 실험 실행
# ---------------------------------------------------------
print("📊 PNA scalers 설정을 위한 Degree Histogram 계산 중...")
deg = degree(graph_data.edge_index[0], num_nodes=graph_data.x.size(0))
deg_train = deg[graph_data.train_mask].to(torch.long)
deg_hist = torch.zeros(int(deg_train.max().item()) + 1)
deg_hist.scatter_add_(0, deg_train, torch.ones_like(deg_train, dtype=torch.float))

experiments = {
    "Single_MAX (기존 챔피언)": GraphSAGE_Single_MAX(graph_data.num_features, 64),
    "PNA (고도화 아키텍처)": GraphSAGE_PNA(graph_data.num_features, 64, deg_hist)
}

results = []
# (이후 학습 및 evaluate_model 로직 실행)

📊 PNA scalers 설정을 위한 Degree Histogram 계산 중...


In [9]:
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
import pandas as pd

# 1. 날짜 분석을 위해 테스트 구간의 Polars 데이터 확보
# val_cutoff는 이전 코드에서 정의된 값을 사용합니다.
test_full = df_hybrid.filter(pl.col("time_group") >= val_cutoff)
y_true = test_pd['is_laundering'].values

def analyze_pna_performance(y_true, y_prob, test_df_pl):
    # 임계값 설정 (수사 우선순위 기준 0.7 사용)
    threshold = 0.7 
    y_pred = (y_prob >= threshold).astype(int)
    
    print("\n" + "="*80)
    print(f"📊 [PNA 앙상블] 최종 성능 상세 분석 (Threshold: {threshold})")
    print("="*80)
    
    # 1) 전체 메트릭 (F1, Precision, Recall)
    print("\n[1] Classification Report:")
    print(classification_report(y_true, y_pred, digits=4))
    
    # 2) 전체 Top-K 탐지 성능 (100, 500, 1000, 5000)
    eval_df = pd.DataFrame({'prob': y_prob, 'actual': y_true})
    sorted_df = eval_df.sort_values(by='prob', ascending=False).reset_index(drop=True)
    
    print("\n[2] 전체 구간 Top-K 검거 성능:")
    print("-" * 55)
    total_frauds = y_true.sum()
    for k in [100, 500, 1000, 5000]:
        if len(sorted_df) < k: continue
        hits = int(sorted_df.head(k)['actual'].sum())
        prec = (hits / k) * 100
        rec = (hits / total_frauds) * 100
        print(f"Top {k:<5} | 검거: {hits:<5} | 정밀도: {prec:>6.2f}% | 재현율: {rec:>6.2f}%")

    # 3) 일자별 Top-100 검거 추이
    eval_df['date'] = test_df_pl['time_group'].dt.date().to_pandas()
    daily_stats = []
    
    for d in sorted(eval_df['date'].unique()):
        day_data = eval_df[eval_df['date'] == d]
        if len(day_data) < 100: continue
        
        day_hits = int(day_data.sort_values(by='prob', ascending=False).head(100)['actual'].sum())
        daily_stats.append({'Date': d, 'Hits@100': day_hits})
    
    print("\n[3] 일자별 Top-100 검거 추이:")
    print("-" * 55)
    print(pd.DataFrame(daily_stats).to_string(index=False))

# 실행
analyze_pna_performance(y_true, y_prob, test_full)


📊 [PNA 앙상블] 최종 성능 상세 분석 (Threshold: 0.7)

[1] Classification Report:
              precision    recall  f1-score   support

       False     0.9988    0.9938    0.9963   5772252
        True     0.2820    0.6784    0.3984     20661

    accuracy                         0.9927   5792913
   macro avg     0.6404    0.8361    0.6974   5792913
weighted avg     0.9963    0.9927    0.9942   5792913


[2] 전체 구간 Top-K 검거 성능:
-------------------------------------------------------
Top 100   | 검거: 100   | 정밀도: 100.00% | 재현율:   0.48%
Top 500   | 검거: 493   | 정밀도:  98.60% | 재현율:   2.39%
Top 1000  | 검거: 976   | 정밀도:  97.60% | 재현율:   4.72%
Top 5000  | 검거: 4585  | 정밀도:  91.70% | 재현율:  22.19%

[3] 일자별 Top-100 검거 추이:
-------------------------------------------------------
      Date  Hits@100
2022-09-14        95
2022-09-15        95
2022-09-16        86
2022-09-17       100
2022-09-18       100
2022-09-19       100
2022-09-20       100
2022-09-21       100
2022-09-22       100
2022-09-23       100
2022